# Topic Recommendation based on content view


### Getting the Dataset

In [ ]:
pip install implicit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 37.6 MB/s eta 0:00:00


In [38]:
from pandas.io.parsers.readers import csv
from google.colab import drive
import pandas as pd
from scipy.sparse import coo_matrix
import numpy as np
import os

In [39]:
#mounting the drive

drive.mount('/content/drive')

os.getcwd()

os.chdir("/content/drive/MyDrive/abw_project_personalization/Result")
os.getcwd()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


'/content/drive/MyDrive/abw_project_personalization/Result'

In [40]:
df = pd.read_csv('updated_contents.csv')

In [41]:
df.info ()
df.head ()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2018 entries, 0 to 2017
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   user_id     2018 non-null   int64
 1   content_id  2018 non-null   int64
dtypes: int64(2)
memory usage: 31.7 KB


,user_id,content_id
0,60737,13
1,60737,8
2,60737,5
3,60737,5
4,60737,2


In [42]:
# Determine the unique user_id and content_id values
unique_users = df['user_id'].unique()
unique_contents = df['content_id'].unique()

# Map the user_ids and content_ids to indices
user_to_index = {user_id: index for index, user_id in enumerate(unique_users)}
content_to_index = {content_id: index for index, content_id in enumerate(unique_contents)}

df['user_index'] = df['user_id'].map(user_to_index)
df['content_index'] = df['content_id'].map(content_to_index)

view_counts = df.groupby(['user_index', 'content_index']).size().reset_index(name='view_count')

content_user_views = coo_matrix(
    (view_counts['view_count'], (view_counts['content_index'], view_counts['user_index'])),
    shape=(len(unique_contents), len(unique_users))
)

# The arrays for users and artists
users = np.array(unique_users, dtype=object)
contents = np.array(unique_contents, dtype=object)

### Training a Model

Implicit provides implementations of several different algorithms for implicit feedback recommender systems. For this project, I will be looking at the `AlternatingLeastSquares` model that's based off the paper  [Collaborative Filtering for Implicit Feedback Datasets](http://yifanhu.net/PUB/cf.pdf). This model aims to learn a binary target of whether each user has interacted with each item - but weights each binary interaction by a confidence value of how confident we are in this user/item interaction. The implementation in implicit uses the values of a sparse matrix to represent the confidences, with the non zero entries representing whether or not the user has interacted with the item.

The first step in using this model is going to be transforming the raw view counts from the original dataset into values that can be used as confidences. We want to give repeated views more confidence in the model, but have this effect taper off as the number of repeated views increases to reduce the impact a single superfan has on the model. Likewise we want to direct some of the confidence weight away from popular items. To do this we'll use a [bm25](https://en.wikipedia.org/wiki/Okapi_BM25) weighting scheme inspired from classic information retrieval:

In [43]:
from implicit.nearest_neighbours import bm25_weight

# weight the matrix, both to reduce impact of users that have viewed the same content thousands of times
# and to reduce the weight given to popular items
content_user_views = bm25_weight(content_user_views, K1=100, B=0.8)

# get the transpose since the most of the functions in implicit expect (user, item) sparse matrices instead of (item, user)
user_views = content_user_views.T.tocsr()

Once we have a weighted confidence matrix, we can use that to train an ALS model using implicit:

In [44]:
from implicit.als import AlternatingLeastSquares

model = AlternatingLeastSquares(factors=64, regularization=0.05, alpha=2.0)
model.fit(user_views)

  0%|          | 0/15 [00:00<?, ?it/s]

### Making Recommendations

After training the model, you can make recommendations for either a single user or a batch of users with the `.recommend` function on the model:

In [45]:
print(users.shape)

(388,)


In [48]:
df.head (100)

,user_id,content_id,user_index,content_index
0,60737,13,0,0
1,60737,8,0,1
2,60737,5,0,2
3,60737,5,0,2
4,60737,2,0,3
...,...,...,...,...
95,1327512,1458,13,39
96,1327512,1461,13,40
97,1327512,1459,13,38
98,1327512,1460,13,37


In [46]:
# Get recommendations for the a single user
userid = 1
ids, scores = model.recommend(userid, user_views[userid], N=10, filter_already_liked_items=False)

The `.recommend` call will compute the `N` best recommendations for each user in the input, and return the itemids in the `ids` array as well as the computed scores in the `scores` array. We can see what the contents are recommended for each user by looking up the ids in the `contents` array:

In [50]:
# Use pandas to display the output in a table, pandas isn't a dependency of implicit otherwise
pd.DataFrame({"content": contents[ids], "score": scores, "already_liked": np.in1d(ids, user_views[userid].indices)})

,content,score,already_liked
0,415,1.004459,True
1,373,1.003295,True
2,382,1.000572,True
3,418,0.999752,True
4,334,0.999730,True
5,337,0.999726,True
6,1514,0.999725,True
7,412,0.999706,True
8,367,0.999705,True
9,1512,0.999703,True


The `already_liked` column there shows if the user has interacted with the item already, and in this result most of the items being returned have already been interacted with by the user. We can remove these items from the result set with the `filter_already_liked_items` parameter - setting to `True` will remove all of these items from the results. The `user_views[userid]` parameter is used to look up what items each user has interacted with, and can just be set to None if you aren't filtering the users own likes or recalculating the user representation on the fly.

There are also more filtering options present in the `filter_items` parameter and `items` parameter, as well as options for recalculating the user representation on the fly with the `recalculate_user` parameter. See the API reference for more details.

### Recommending similar items

Each model in implicit also has the ability to show related items through the `similar_items` method. For instance to get the related items for the Beatles:

In [49]:
print(contents.shape)

(179,)


In [ ]:
# get related items for the content
ids, scores= model.similar_items(1)

# display the results using pandas for nicer formatting
pd.DataFrame({"content": contents[ids], "score": scores})

,content,score
0,1236,1.000000
1,8,0.999980
2,1237,0.999978
3,4,0.773201
4,14,0.668744
5,123,0.654801
6,121,0.654799
7,11,0.630939
8,2,0.524676
9,5,0.519718


### Making batch recommendations

The `.recommend`, `.similar_items` and `.similar_users` calls all have the ability to generate batches of recommendations - in addition to just calculating a single user or item.  Passing an array of userids or itemids to these methods will trigger the batch methods, and return a 2D array of ids and scores - with each row in the output matrices corresponding to value in the input. This will tend to be quite a bit more efficient calling the method repeatedly, as implicit will use multiple threads on the CPU and achieve better device utilization on the GPU with larger batches.

In [ ]:
# Make recommendations for the first 100 users in the dataset
userids = np.arange(100)
ids, scores = model.recommend(userids, user_views[userids])
ids, ids.shape

(array([[ 69, 175,  54,  70,  85, 176,  86, 128, 172, 177],
        [175,  71,  35,  90,   2, 176, 155,  64, 162,  63],
        [162, 133, 173, 158, 135,  27, 187,  68, 116,  30],
        [133, 186,  23, 134, 129,  54,  47, 101,  62,  91],
        [159, 181, 179, 162, 126, 128, 125,  69, 157, 124],
        [159, 181, 156, 133, 154, 153, 129, 122, 172, 126],
        [ 37, 117, 151, 132, 166,  67, 187, 165,  78, 157],
        [122, 121, 173,  70, 178, 180,  52,  77, 159,  28],
        [132, 158, 162,  75, 134,  53,  70, 119,  78,  27],
        [126, 187,  39, 135,  56, 125,  70, 116,  23, 163],
        [172, 179, 130, 187,  77, 115, 153, 154, 155, 134],
        [ 53,  32, 151, 135,  75, 158,  92,  27, 136, 134],
        [172, 118, 126, 187, 115,  75,  23, 163, 152, 151],
        [162, 133, 173, 158, 135,  27, 187,  68, 116,  30],
        [ 27,  67, 132, 162,  56, 135,  14, 166,  77, 179],
        [152, 135, 134, 162, 173, 186, 133, 158, 120, 118],
        [123, 181, 188, 140, 109, 162,  